# Validación externa — Predicciones vs etiquetas GHS (PubChem)

Las etiquetas **GHS** (Globally Harmonized System) de PubChem son datos regulatorios
independientes del entrenamiento del modelo. Este notebook analiza:

1. **Cobertura GHS** en el corpus panameño
2. **Distribución** de peligros (oral, endocrino, genotóxico, acuático)
3. **Perfil por familia** química y por ingrediente MIDA
4. **Correlación** con predicciones del modelo GNN-GIN (si existen)

**Requisitos:**
- `data/raw/pubchem_ghs_labels.csv` → `make build-panama-corpus`
- `outputs/results/panama_predictions.csv` → `make explain-panama` (opcional, sección 5)

## 0. Configuración

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from src.data.dataset import TASK_NAMES
from src.data.pubchem_api import MIDA_ACTIVE_INGREDIENTS, GHS_HAZARD_CODES

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 120})

CIDS_CSV = ROOT / "data" / "raw" / "pubchem_panama_cids.csv"
GHS_CSV = ROOT / "data" / "raw" / "pubchem_ghs_labels.csv"
PRED_CSV = ROOT / "outputs" / "results" / "panama_predictions.csv"
FIG_DIR = ROOT / "outputs" / "panama" / "ghs"
REPORT_DIR = ROOT / "outputs" / "reports"
FIG_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

BINARY_COLS = ["toxic_oral", "endocrine_risk", "genotoxic", "aquatic_tox"]
BINARY_LABELS = {
    "toxic_oral": "Tóxico oral (H300-H302)",
    "endocrine_risk": "Riesgo reproductivo (H360-H361)",
    "genotoxic": "Mutagénico/carcinogénico (H340-H351)",
    "aquatic_tox": "Toxicidad acuática (H400-H412)",
}

print(f"Raíz: {ROOT}")

## 1. Carga y merge corpus + GHS

In [ ]:
if not GHS_CSV.is_file():
    raise FileNotFoundError(
        f"No existe {GHS_CSV}. Ejecuta: make build-panama-corpus"
    )
if not CIDS_CSV.is_file():
    raise FileNotFoundError(
        f"No existe {CIDS_CSV}. Ejecuta: make build-panama-corpus"
    )

cids_df = pd.read_csv(CIDS_CSV)
ghs_df = pd.read_csv(GHS_CSV)

merged = cids_df.merge(ghs_df, on="CID", how="left")
merged["has_ghs"] = merged["ghs_codes"].fillna("").astype(str).str.len() > 0
merged["is_mida"] = merged["source"] == "MIDA_name_search"

print(f"Corpus: {len(cids_df)} compuestos")
print(f"GHS:    {len(ghs_df)} registros")
print(f"Con al menos un código H: {merged['has_ghs'].sum()} ({merged['has_ghs'].mean():.1%})")
display(merged[["name", "CID", "family", "ghs_codes"] + BINARY_COLS].head(10))

## 2. Distribución de peligros GHS

In [ ]:
rates = merged[BINARY_COLS].mean().rename("fración_positiva")
rates.index = [BINARY_LABELS.get(c, c) for c in rates.index]
display(rates.to_frame().round(3))

fig, ax = plt.subplots(figsize=(9, 4))
rates.plot(kind="bar", ax=ax, color=["#d62728", "#ff7f0e", "#9467bd", "#1f77b4"])
ax.set_ylabel("Fracción de compuestos con etiqueta")
ax.set_title("Prevalencia de categorías GHS en el corpus panameño")
ax.set_xticklabels(rates.index, rotation=25, ha="right")
plt.tight_layout()
fig.savefig(FIG_DIR / "ghs_prevalence.png", bbox_inches="tight")
plt.show()

## 3. Códigos H individuales

In [ ]:
code_counts: dict[str, int] = {}
for codes in merged["ghs_codes"].fillna(""):
    for code in str(codes).split("|"):
        code = code.strip()
        if code:
            code_counts[code] = code_counts.get(code, 0) + 1

if code_counts:
    h_df = (
        pd.Series(code_counts, name="n")
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={"index": "codigo_H"})
    )
    h_df["descripcion"] = h_df["codigo_H"].map(GHS_HAZARD_CODES).fillna("(otro)")
    display(h_df.head(20))

    fig, ax = plt.subplots(figsize=(10, max(4, 0.3 * min(20, len(h_df)))))
    top = h_df.head(15)
    ax.barh(top["codigo_H"], top["n"], color="steelblue")
    ax.set_xlabel("Número de compuestos")
    ax.set_title("Códigos H más frecuentes")
    ax.invert_yaxis()
    plt.tight_layout()
    fig.savefig(FIG_DIR / "ghs_h_codes.png", bbox_inches="tight")
    plt.show()
else:
    print(
        "No se encontraron códigos H en el CSV. "
        "PubChem puede no exponer GHS para estos CIDs vía la API PUG REST."
    )

## 4. GHS por familia química

In [ ]:
fam_ghs = (
    merged.groupby("family")[BINARY_COLS + ["has_ghs"]]
    .mean()
    .sort_values("genotoxic", ascending=False)
)
display(fam_ghs.round(3))

plot_fam = fam_ghs[BINARY_COLS].copy()
plot_fam.columns = [BINARY_LABELS[c] for c in plot_fam.columns]

plt.figure(figsize=(12, max(5, 0.4 * len(plot_fam))))
sns.heatmap(plot_fam, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0, vmax=1, linewidths=0.5)
plt.title("Fracción con etiqueta GHS por familia")
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()
plt.savefig(FIG_DIR / "ghs_by_family_heatmap.png", bbox_inches="tight")
plt.show()

## 5. Ingredientes activos MIDA

In [ ]:
mida_lower = {n.lower() for n in MIDA_ACTIVE_INGREDIENTS}
mida = merged[merged["name"].str.lower().isin(mida_lower)].copy()
mida = mida.sort_values("name")

cols_show = ["name", "CID", "family", "ghs_codes"] + BINARY_COLS
display(mida[cols_show])

if mida["has_ghs"].any():
    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(mida))))
    mida_plot = mida.set_index("name")[BINARY_COLS]
    mida_plot.columns = [BINARY_LABELS[c] for c in mida_plot.columns]
    sns.heatmap(mida_plot, annot=True, fmt="d", cmap="Reds", cbar=False, linewidths=0.5, ax=ax)
    ax.set_title("Etiquetas GHS binarias — ingredientes MIDA")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "ghs_mida_heatmap.png", bbox_inches="tight")
    plt.show()
else:
    print("Sin códigos GHS para ingredientes MIDA en PubChem (común con la API actual).")

## 6. Correlación con predicciones del modelo

Compara scores compuestos del modelo con flags GHS binarios (validación externa).

In [ ]:
if not PRED_CSV.is_file():
    print(f"No hay predicciones en {PRED_CSV}")
    print("Ejecuta: make explain-panama")
else:
    pred_df = pd.read_csv(PRED_CSV)
    val = pred_df.merge(ghs_df, left_on="cid", right_on="CID", how="inner")

    val["pred_endocrine"] = val[["NR-AR", "NR-ER", "NR-ER-LBD"]].max(axis=1)
    val["pred_genotox"] = val[["SR-p53", "SR-AtAD5"]].max(axis=1)
    val["pred_oxidative"] = val["SR-ARE"]

    pairs = [
        ("pred_endocrine", "endocrine_risk", "Endocrino vs H360/H361"),
        ("pred_genotox", "genotoxic", "Genotóxico vs H340/H350"),
        ("pred_oxidative", "toxic_oral", "SR-ARE vs H300-H302"),
    ]

    print("Correlación Spearman (predicción continua vs GHS binario):\n")
    corr_rows = []
    for pred_col, ghs_col, label in pairs:
        sub = val[[pred_col, ghs_col]].dropna()
        if sub[ghs_col].nunique() < 2:
            rho, n = np.nan, len(sub)
            note = "sin variación GHS"
        else:
            rho = float(sub[pred_col].corr(sub[ghs_col], method="spearman"))
            n = len(sub)
            note = ""
        print(f"  {label}: ρ = {rho:.3f} (n={n}) {note}")
        corr_rows.append({"comparison": label, "spearman_rho": rho, "n": n, "note": note})

    out_path = REPORT_DIR / "ghs_validation_summary.csv"
    pd.DataFrame(corr_rows).to_csv(out_path, index=False)
    print(f"\nResumen guardado: {out_path}")

    # Scatter solo si hay variación en GHS
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, (pred_col, ghs_col, label) in zip(axes, pairs):
        sub = val[[pred_col, ghs_col, "compuesto"]].dropna()
        if sub[ghs_col].nunique() < 2:
            ax.text(0.5, 0.5, "Sin variación GHS", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(label)
            continue
        jitter = np.random.default_rng(0).uniform(-0.05, 0.05, size=len(sub))
        ax.scatter(sub[pred_col], sub[ghs_col] + jitter, alpha=0.6, s=40)
        ax.set_xlabel(pred_col)
        ax.set_ylabel(ghs_col)
        ax.set_title(label)
        ax.set_ylim(-0.2, 1.2)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "pred_vs_ghs_scatter.png", bbox_inches="tight")
    plt.show()

    # MIDA: tabla cruzada predicción vs GHS
    mida_val = val[val["compuesto"].str.lower().isin(mida_lower)]
    if len(mida_val):
        display(
            mida_val[["compuesto", "tarea_critica", "prob_max", "alerta", "ghs_codes"] + BINARY_COLS]
            .sort_values("prob_max", ascending=False)
        )

## 7. Exportar tabla de validación

In [ ]:
export_cols = [
    "name", "CID", "family", "ghs_codes",
    "toxic_oral", "endocrine_risk", "genotoxic", "aquatic_tox", "has_ghs",
]
export_cols = [c for c in export_cols if c in merged.columns]
out = REPORT_DIR / "ghs_corpus_profile.csv"
merged[export_cols].to_csv(out, index=False)
print(f"Perfil GHS del corpus: {out}")
print(f"Figuras: {FIG_DIR}")
print("\nTambién puedes ejecutar: make validate-ghs")